# E3 — 재순위(R)·LLM 판정 구조(J, LangGraph)·일관성 검사

Colab(T4 GPU) 전용. **dev 내보내기 파일만** 받는다. 정답 값은 읽지도 다루지도 않고, 정확도도 여기서 계산하지 않는다(PC 의 `app.evaluation.run` 에서만).

셀 순서: 0 설치·GPU·시드 → 1 업로드·sha256 → 2 R 재순위 → 3 J 판정 정의(LangGraph) → 4 일관성 검사(5회) → 5 스키마 검증·저장.
업로드할 파일: `dev-C2DV.json`(필수), `dev-C1GDV.json`(재순위만 필요하면), `e3_core.py`, `external-result.schema.json`.

In [ ]:
# 셀 0 — 설치(버전 고정)·GPU 확인·시드 고정
# 2026-10 Colab(Python 3.13·CUDA 13)에서 옛 bitsandbytes 0.44.1 은 GPU 지원 없이 빌드되고 triton.ops 오류가 난다 → 최소 버전만 지정하고 설치된 실제 버전을 기록한다
!pip -q install -U "transformers>=4.51,<5" "huggingface-hub<1.0" "accelerate>=1.1" "bitsandbytes>=0.46.1" "langgraph>=0.2.53" "jsonschema>=4.23"
# transformers 5.x 는 이 노트북의 모델 로딩 경로와 맞지 않아 4.x 로 제한한다. 설치가 끝나면 반드시 "런타임 > 세션 다시 시작" 후 이 셀을 다시 실행한다. (설치 직후 같은 세션에서 바로 이어 쓰면 옛 모듈과 섞여 ImportError 가 난다)
# 설치가 끝난 뒤 이 셀 아래에서 "런타임 > 세션 다시 시작"이 필요하다는 안내가 뜨면 다시 시작한 뒤 셀 0 을 한 번 더 실행한다(두 번째는 설치가 바로 끝난다)

import random
import numpy as np
import torch

assert torch.cuda.is_available(), "런타임 > 런타임 유형 변경 > T4 GPU 로 바꾼 뒤 다시 실행하세요"
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

from importlib.metadata import version
VERSIONS = {n: version(n) for n in ("transformers", "accelerate", "bitsandbytes", "langgraph", "jsonschema", "torch")}
print("설치된 버전(결과 params에 기록):", VERSIONS)

In [ ]:
# 셀 1 — 업로드·sha256 확인
import hashlib, json, os, sys
from google.colab import files

os.makedirs("results", exist_ok=True)
uploaded = files.upload()
# Colab 은 같은 이름이 이미 있으면 "이름 (1).확장자" 로 저장한다 → 원래 이름으로 되돌린다(새로 올린 것이 이긴다)
import re
for n in list(uploaded):
    base = re.sub(r" \(\d+\)(?=\.[^.]+$)", "", n)
    if base != n:
        os.replace(n, base)
        uploaded[base] = uploaded.pop(n)
# 이미 올려 둔 파일이 있으면(업로드를 건너뛴 경우) 그것도 인정한다
for n in os.listdir("."):
    uploaded.setdefault(n, None)  # dev-C2DV.json, dev-C1GDV.json(선택), e3_core.py, external-result.schema.json
sys.path.insert(0, ".")
import e3_core

# docs/LCA/reviews/E2b-run.md 에 적힌 값. 다르면 다른 파일이므로 중단한다
EXPECTED_SHA256 = {
    "dev-C2DV.json": "f21a2c4104b8824994489f07d58ca6252d8e5ebdee51fa1039238b6040303ebf",
    "dev-C1GDV.json": "1bea380a92790dd9922c2f436d2c92582e9d81a4eb1879dbefbcc822bd2f7939",
}
EXPORTS = {}  # 이름(확장자 없음) -> (내보내기 dict, sha256)
for name in sorted(n for n in uploaded if n in EXPECTED_SHA256):
    doc, sha = e3_core.load_export(name)  # 정답 관련 키가 있으면 여기서 거부
    ok = sha == EXPECTED_SHA256[name]
    print(f"{name}: sha256 {sha}  {'OK' if ok else '*** E2b-run.md 의 값과 다릅니다 ***'}  행 {len(doc['rows'])}  split {doc['split']}  base {doc['base_case']}")
    assert ok, f"{name} 이 내보낸 파일과 다릅니다. 업로드한 파일을 확인하세요(이름에 ' (1)' 이 붙었다면 지우고 다시 올리세요)"
    assert doc["split"] == "dev", "dev 내보내기만 받습니다"
    EXPORTS[name.removesuffix(".json")] = (doc, sha)
assert EXPORTS, "dev-C2DV.json 또는 dev-C1GDV.json 을 올려야 합니다"
SCHEMA = json.load(open("external-result.schema.json", encoding="utf-8"))
print("내보내기:", list(EXPORTS))

In [ ]:
# 셀 2 — R 재순위: 교차 인코더(eval·no_grad·fp16·배치 고정). 내보내기 파일마다 결과 1개
import time
from transformers import AutoModelForSequenceClassification, AutoTokenizer

RERANK_MODEL = "BAAI/bge-reranker-v2-m3"   # 바꿀 수 있다
MAX_LENGTH = 512
BATCH = 10                                  # 후보 30개를 10개씩 3번(고정)
DTYPE = "float16"

rtok = AutoTokenizer.from_pretrained(RERANK_MODEL)
rmodel = AutoModelForSequenceClassification.from_pretrained(RERANK_MODEL, torch_dtype=torch.float16).cuda().eval()


def rerank_row(row):
    query = e3_core.query_text(row)
    texts = [e3_core.candidate_text(c) for c in row["candidates"]]
    scores = []
    with torch.no_grad():
        for i in range(0, len(texts), BATCH):
            chunk = texts[i:i + BATCH]
            enc = rtok([query] * len(chunk), chunk, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to("cuda")
            scores += rmodel(**enc).logits.view(-1).float().cpu().tolist()
    return e3_core.rank_by_scores(row["candidates"], scores)


RERANK = {}  # 내보내기 이름 -> 스키마 형식 결과
for ename, (doc, sha) in EXPORTS.items():
    per_row, t0 = [], time.time()
    for row in doc["rows"]:
        t = time.perf_counter()
        ranking = rerank_row(row)
        per_row.append({"label_id": row["label_id"], "ranking": ranking, "latency_ms": int((time.perf_counter() - t) * 1000)})
    meta = {"kind": "rerank", "model": RERANK_MODEL, "structure": None, "prompt_sha256": None,
            "params": {"model": RERANK_MODEL, "max_length": MAX_LENGTH, "batch": BATCH, "dtype": DTYPE, "top_k": len(doc["rows"][0]["candidates"])}}
    RERANK[ename] = e3_core.make_result(doc, sha, meta, per_row)
    ms = [x["latency_ms"] for x in per_row]
    print(f"{ename}: R 완료 {len(per_row)}행, {time.time() - t0:.0f}초, 행당 {sum(ms) / len(ms):.0f}ms")

del rmodel
torch.cuda.empty_cache()

In [ ]:
# 셀 3 — J 판정(LangGraph) 정의: 입력은 R 결과 상위 10개. 로컬 오픈 모델 4bit, greedy
import time
from typing import Optional, TypedDict

from google.colab import drive
from langgraph.graph import END, START, StateGraph
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

JUDGE_MODEL = "Qwen/Qwen2.5-7B-Instruct"   # 바꿀 수 있다
MAX_NEW_TOKENS = 64                          # 고정
JUDGE_EXPORTS = ["dev-C2DV"]                 # 판정은 dev-C2DV 하나에서만 먼저(C1 기준은 R 결과를 보고 정한다)
SEEDS = (0, 1, 2, 3, 4)

drive.mount("/content/drive")
CACHE_PATH = "/content/drive/MyDrive/lci_e3_cache/judge_cache.jsonl"
os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
CACHE = {}  # cache_key(model, prompt_sha256, label_id, 순서) -> {"key", "text", "ms"}
if os.path.exists(CACHE_PATH):
    for line in open(CACHE_PATH, encoding="utf-8"):
        rec = json.loads(line)
        CACHE[rec["key"]] = rec
print("캐시", len(CACHE), "건")

jtok = AutoTokenizer.from_pretrained(JUDGE_MODEL)
jmodel = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL, device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)).eval()


def generate(prompt):
    text = jtok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
    enc = jtok(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = jmodel.generate(**enc, do_sample=False, temperature=None, top_p=None, top_k=None, max_new_tokens=MAX_NEW_TOKENS,
                              pad_token_id=jtok.eos_token_id)
    return jtok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)


def call(structure, step, label_id, order, prompt):
    """캐시를 거치는 모델 호출 -> (출력, 생성 ms). 키 = (모델, 프롬프트 sha256, label_id, 단계+후보 순서)."""
    key = e3_core.cache_key(JUDGE_MODEL, e3_core.prompt_sha256(structure), label_id, [step, *order])
    if key not in CACHE:
        t = time.perf_counter()
        text = generate(prompt)
        CACHE[key] = {"key": key, "text": text, "ms": int((time.perf_counter() - t) * 1000)}
        with open(CACHE_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(CACHE[key], ensure_ascii=False) + "\n")
    return CACHE[key]["text"], CACHE[key]["ms"]


class State(TypedDict, total=False):
    row: dict
    cands: list          # R 상위 10개 후보(R 순서)
    seed: int
    role: str
    decision: str        # pick | none
    picked: Optional[str]
    reason: str
    invalid: int         # 판정 출력 형식 오류(none 으로 기록하고 센다)
    ms: int


def _judge(state, structure, cands, **fields):
    shown = e3_core.permute(cands, state["seed"])  # 보여 주는 순서만 섞는다
    prompt = e3_core.build_judge_prompt(state["row"], shown, structure, "judge", **fields)
    text, ms = call(structure, "judge", state["row"]["label_id"], [c["dataset_id"] for c in shown], prompt)
    kind, picked = e3_core.parse_judge_output(text, [c["dataset_id"] for c in shown])
    return {"decision": "pick" if kind == "pick" else "none", "picked": picked, "reason": text.strip()[:e3_core.REASON_MAX],
            "invalid": state.get("invalid", 0) + (kind == "invalid"), "ms": state.get("ms", 0) + ms}


def s1_judge(state):
    return _judge(state, "S1", state["cands"])


def s2_role(state):
    prompt = e3_core.build_judge_prompt(state["row"], [], "S2", "role")
    text, ms = call("S2", "role", state["row"]["label_id"], [], prompt)
    return {"role": e3_core.parse_role(text), "ms": state.get("ms", 0) + ms}


def s2_judge(state):
    cands, _ = e3_core.filter_by_role(state["cands"], state["role"])
    return _judge(state, "S2", cands, role=state["role"])


def s3_judge(state):
    return _judge(state, "S3", state["cands"])


def s3_verify(state):
    if state["decision"] != "pick":
        return {}
    c = next(c for c in state["cands"] if c["dataset_id"] == state["picked"])
    prompt = e3_core.build_judge_prompt(state["row"], [], "S3", "verify", picked=f"{c['dataset_id']}: {e3_core.candidate_text(c)}")
    text, ms = call("S3", "verify", state["row"]["label_id"], [state["picked"]], prompt)
    out = {"ms": state.get("ms", 0) + ms}
    if e3_core.parse_yes_no(text) is True:      # 모순 -> none (형식이 다르면 판정을 유지)
        out.update(decision="none", picked=None)
    return out


def build_graph(structure):
    g = StateGraph(State)
    nodes = {"S1": [("judge", s1_judge)], "S2": [("role", s2_role), ("judge", s2_judge)], "S3": [("judge", s3_judge), ("verify", s3_verify)]}[structure]
    for name, fn in nodes:
        g.add_node(name, fn)
    g.add_edge(START, nodes[0][0])
    for (a, _), (b, _) in zip(nodes, nodes[1:]):
        g.add_edge(a, b)
    g.add_edge(nodes[-1][0], END)
    return g.compile()


GRAPHS = {s: build_graph(s) for s in e3_core.STRUCTURES}


def run_structure(doc, r_result, structure, seed):
    """한 회차: 모든 행을 판정 -> (결과 행 목록, invalid 개수). 결과 ranking = picked 를 1위로 올리고 나머지는 R 순서."""
    rmap = {x["label_id"]: x for x in r_result["results"]}
    per_row, invalid = [], 0
    for row in doc["rows"]:
        ranking = rmap[row["label_id"]]["ranking"]
        by_id = {c["dataset_id"]: c for c in row["candidates"]}
        cands = [by_id[d] for d, _ in ranking[:e3_core.TOP_JUDGE]]
        out = GRAPHS[structure].invoke({"row": row, "cands": cands, "seed": seed, "invalid": 0, "ms": 0})
        invalid += out["invalid"]
        per_row.append({"label_id": row["label_id"], "ranking": e3_core.rank_with_pick(ranking, out["picked"]), "decision": out["decision"],
                        "picked_dataset_id": out["picked"], "reason": out["reason"], "latency_ms": out["ms"]})
    return per_row, invalid

In [ ]:
# 셀 4 — 일관성 검사: 구조마다 permute 시드 0~4 로 5회. flip_rate 가 5% 를 넘으면 탈락(_UNSTABLE)
JUDGE = {}  # (내보내기 이름, 구조) -> {"result": 시드 0 결과, "flip": 비율, "unstable": bool}
t_all = time.time()
for ename in JUDGE_EXPORTS:
    doc, sha = EXPORTS[ename]
    for structure in e3_core.STRUCTURES:
        t0, runs, invalid = time.time(), {}, {}
        base_meta = {"kind": "judge", "model": JUDGE_MODEL, "structure": structure, "prompt_sha256": e3_core.prompt_sha256(structure)}
        for seed in SEEDS:
            per_row, invalid[seed] = run_structure(doc, RERANK[ename], structure, seed)
            runs[seed] = per_row
        results = [e3_core.make_result(doc, sha, {**base_meta, "params": {}}, runs[s]) for s in SEEDS]
        flip = e3_core.flip_rate(results)
        unstable = e3_core.is_unstable(flip)
        params = {"model": JUDGE_MODEL, "quantization": "4bit-nf4", "temperature": 0, "do_sample": False, "max_new_tokens": MAX_NEW_TOKENS,
                  "top_k": e3_core.TOP_JUDGE, "rerank_model": RERANK[ename]["run"]["model"], "seeds": list(SEEDS), "flip_rate": flip,
                  "flip_max": e3_core.FLIP_MAX, "unstable": unstable, "invalid_count": {str(s): invalid[s] for s in SEEDS},
                  "other_seed_decisions": {str(s): e3_core.decision_summary(runs[s]) for s in SEEDS if s != 0},
                  "versions": VERSIONS}
        JUDGE[(ename, structure)] = {"result": e3_core.make_result(doc, sha, {**base_meta, "params": params}, runs[0]), "flip": flip, "unstable": unstable}
        ms = [x["latency_ms"] for x in runs[0]]
        print(f"{ename} {structure}: flip_rate {flip:.1%} {'탈락(_UNSTABLE)' if unstable else '통과'} | invalid {sum(invalid.values())} | "
              f"{time.time() - t0:.0f}초 | 행당 생성 {sum(ms) / len(ms):.0f}ms(시드 0)")
print(f"전체 {time.time() - t_all:.0f}초")

In [ ]:
# 셀 5 — 스키마 검증 후 저장·내려받기: results/E3-<export>-<kind>-<structure>.json
from jsonschema import Draft202012Validator

validator = Draft202012Validator(SCHEMA)
outputs = [(ename, "rerank", None, False, res) for ename, res in RERANK.items()]
outputs += [(ename, "judge", structure, v["unstable"], v["result"]) for (ename, structure), v in JUDGE.items()]
for ename, kind, structure, unstable, result in outputs:
    errors = sorted(validator.iter_errors(result), key=lambda e: list(e.path))
    assert not errors, f"{ename} {kind} {structure}: 스키마 오류 {errors[0].message}"
    path = os.path.join("results", e3_core.result_filename(ename, kind, structure, unstable))
    with open(path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False)
    print(("탈락 " if unstable else "") + path, os.path.getsize(path), "바이트")
    files.download(path)